# model-save-state-dict — worked example 1: Rank-0-only save then barrier

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `model-save-state-dict`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In DDP all replicas hold identical parameters, so the checkpoint is written by rank 0 only: `if rank == 0: t.save(model.state_dict(), path)`. A `dist.barrier()` after the save makes every other rank wait until the file is fully written before any of them reads it.

## Worked solution

We model the multi-rank save with a mock barrier so it runs in one process.

1. **Mock the collective.** `MockDist` records how many times `barrier()` was called — enough to verify the synchronization happened without spawning real processes.
2. **Rank-0 guard.** Only `rank == 0` calls `t.save(model.state_dict(), path)`. The other ranks skip the write because their identical weights would just overwrite the same bytes.
3. **Barrier.** Every rank (including 0) calls `dist.barrier()`. In real DDP this blocks the readers until the writer finishes; here it just increments the counter.
4. **Read back.** After the barrier every rank can `t.load(path)` and see the saved tensor.

The demo simulates a 3-rank world, runs the save worker for each rank, and prints that the file exists, loads back correctly, and the barrier fired once per rank.

In [ ]:
import torch as t
import torch.nn as nn
import tempfile, os

t.manual_seed(0)

class MockDist:
    def __init__(self):
        self.barrier_calls = 0
    def barrier(self):
        self.barrier_calls += 1

def save_worker(rank, dist_mod, model, path):
    if rank == 0:
        t.save(model.state_dict(), path)
    dist_mod.barrier()
    sd = t.load(path, weights_only=True)
    return sd['weight'].sum().item()

model = nn.Linear(2, 2, bias=False)
with t.no_grad():
    model.weight.fill_(7.0)

tmp = tempfile.mkdtemp()
path = os.path.join(tmp, 'ckpt.pt')
dist_mod = MockDist()
world = 3
sums = [save_worker(r, dist_mod, model, path) for r in range(world)]
print('file exists:', os.path.exists(path))
print('per-rank loaded sums:', sums)        # all 28.0 (2x2 of 7s)
print('barriers:', dist_mod.barrier_calls)  # one per rank